# 🚆 Indian Railway Failure Detection ML

## What is this notebook about?

Indian Railways is one of the largest railway networks in the world. Unexpected train failures can cause delays, accidents, and huge costs. In this notebook, we will build a **Machine Learning model** that predicts whether a train needs maintenance or not — **before a failure happens!**

---

## What will we do step by step?

1. 📂 Load and explore the dataset
2. 🔍 Check for missing values
3. 📊 Visualize important patterns
4. 🛠️ Clean and prepare the data
5. 🤖 Train multiple ML models
6. 📈 Compare model performance
7. 🎯 Check feature importance
8. ✅ Conclusion

---

> **Target Variable:** `maintenance_required` — 1 means maintenance is needed, 0 means not needed.

## 📦 Step 1: Import Libraries

We start by importing all the tools (libraries) we need.

- **pandas** — for loading and handling data (like Excel in Python)
- **numpy** — for math operations
- **matplotlib / seaborn** — for making charts and graphs
- **sklearn** — for building ML models
- **xgboost / lightgbm** — powerful and fast ML algorithms

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Make plots look nicer
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print("✅ All libraries imported successfully!")

## 📂 Step 2: Load the Dataset

Let's load our CSV file and take a first look at the data.

In [ ]:
# Load the dataset
df = pd.read_csv('/kaggle/input/indian-railway-failure-detection-maintenance/indian_railway_failure_detection_maintenance_v2.csv')

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Basic info about the dataset
print("Column names and data types:")
df.info()

In [ ]:
# Statistical summary of numeric columns
print("Statistical Summary:")
df.describe().round(2)

## 🎯 Step 3: Understand the Target Variable

Our target column is `maintenance_required`. Let's check how balanced it is.

- **1** → Maintenance Needed
- **0** → No Maintenance Needed

In [ ]:
# Count how many trains need maintenance vs don't
target_counts = df['maintenance_required'].value_counts()
print("Target Class Distribution:")
print(target_counts)
print(f"\nMaintenance needed: {target_counts[1]} ({target_counts[1]/len(df)*100:.1f}%)")
print(f"No maintenance:     {target_counts[0]} ({target_counts[0]/len(df)*100:.1f}%)")

# Plot
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
target_counts.plot(kind='bar', ax=ax[0], color=['steelblue', 'tomato'], edgecolor='black')
ax[0].set_title('Maintenance Required: Class Distribution', fontsize=13)
ax[0].set_xlabel('0 = No Maintenance | 1 = Maintenance Needed')
ax[0].set_ylabel('Count')
ax[0].tick_params(rotation=0)

# Pie chart
ax[1].pie(target_counts, labels=['No Maintenance', 'Maintenance Needed'],
          autopct='%1.1f%%', colors=['steelblue', 'tomato'], startangle=90)
ax[1].set_title('Target Split')

plt.tight_layout()
plt.show()

## 🔍 Step 4: Check for Missing Values

Missing values can cause errors in our model. Let's find them and fix them.

In [ ]:
# Count missing values per column
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print("Columns with missing values:")
print(missing)

# Plot missing values
plt.figure(figsize=(10, 4))
missing.plot(kind='bar', color='coral', edgecolor='black')
plt.title('Missing Values Per Column', fontsize=13)
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 📊 Step 5: Exploratory Data Analysis (EDA)

Let's look at some interesting patterns in the data.

In [ ]:
# Which region has the most maintenance needs?
plt.figure(figsize=(12, 5))
region_maint = df.groupby('region')['maintenance_required'].mean().sort_values(ascending=False)
region_maint.plot(kind='bar', color='teal', edgecolor='black')
plt.title('Maintenance Rate by Railway Region', fontsize=13)
plt.ylabel('Proportion Needing Maintenance')
plt.xlabel('Region')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of key sensor readings
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

key_features = ['wheel_wear_percent', 'brake_pad_wear_percent', 'bearing_temperature_c',
                'axle_temperature_c', 'rail_wear_mm', 'sensor_health_index']

for i, col in enumerate(key_features):
    row, c = divmod(i, 3)
    ax = axes[row][c]
    df[df['maintenance_required']==0][col].dropna().plot(kind='hist', ax=ax, alpha=0.6,
                                                         color='steelblue', label='No Maintenance', bins=30)
    df[df['maintenance_required']==1][col].dropna().plot(kind='hist', ax=ax, alpha=0.6,
                                                         color='tomato', label='Maintenance Needed', bins=30)
    ax.set_title(col.replace('_', ' ').title())
    ax.legend(fontsize=8)

plt.suptitle('Sensor Readings: Maintenance vs No Maintenance', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap — do variables relate to each other?
plt.figure(figsize=(14, 10))
numeric_df = df.select_dtypes(include='number').drop(columns=['train_id'])
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0,
            linewidths=0.5, fmt='.1f')
plt.title('Feature Correlation Heatmap', fontsize=13)
plt.tight_layout()
plt.show()

## 🛠️ Step 6: Data Preprocessing

Before feeding data into ML models, we need to:
1. **Remove leakage columns** — columns that "cheat" by directly revealing the answer (e.g., `failure_type` only exists when maintenance is needed)
2. **Fill missing values** with the median
3. **Encode categorical columns** — convert text to numbers

In [ ]:
# ⚠️ Drop leakage columns — these reveal the target and would give fake high accuracy
# failure_type & failure_severity only exist WHEN maintenance is needed → data leakage!
# risk_score is derived after failure → leakage
# delay_minutes is a consequence of failure → leakage
# train_id is just an identifier

leakage_cols = ['failure_type', 'failure_severity', 'risk_score', 'delay_minutes', 'train_id']
df_clean = df.drop(columns=leakage_cols)

print(f"Columns after removing leakage: {df_clean.shape[1]}")
print("Remaining columns:", df_clean.columns.tolist())

In [ ]:
# Fill missing numeric values with median (middle value — robust to outliers)
num_cols = df_clean.select_dtypes(include='number').columns.tolist()
df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].median())

# Encode categorical columns (text → numbers)
cat_cols = df_clean.select_dtypes(include=['object', 'str']).columns.tolist()
print(f"Categorical columns to encode: {cat_cols}")

le = LabelEncoder()
for col in cat_cols:
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

print("\n✅ Missing values filled and categories encoded!")
print(f"Any missing values left? {df_clean.isnull().sum().sum()}")

In [ ]:
# Split features (X) and target (y)
X = df_clean.drop('maintenance_required', axis=1)
y = df_clean['maintenance_required']

# Split into train (80%) and test (20%) sets
# stratify=y makes sure both sets have similar class ratios
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing  samples : {X_test.shape[0]}")
print(f"Features         : {X_train.shape[1]}")

## 🤖 Step 7: Train Machine Learning Models

We will train **4 models** and compare them:

| Model | Description |
|-------|-------------|
| **Random Forest** | Many decision trees voting together |
| **XGBoost** | Builds trees one by one, each fixing previous mistakes |
| **LightGBM** | Like XGBoost but faster and memory efficient |
| **Gradient Boosting** | Classic boosting algorithm from sklearn |

In [ ]:
# Define all models
models = {
    'Random Forest'     : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost'           : XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', verbosity=0),
    'LightGBM'          : LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
    'Gradient Boosting' : GradientBoostingClassifier(n_estimators=100, random_state=42),
}

# Train each model and record results
results = []

for name, model in models.items():
    print(f"Training {name}...", end=' ')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]  # probability of class 1
    
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    results.append({'Model': name, 'Accuracy': round(acc, 4), 'AUC-ROC': round(auc, 4)})
    print(f"✅ Accuracy: {acc:.4f} | AUC: {auc:.4f}")

results_df = pd.DataFrame(results).sort_values('AUC-ROC', ascending=False)
print("\n=== Model Comparison ===")
print(results_df.to_string(index=False))

## 📊 Step 8: Compare Model Performance

In [ ]:
# Visualize Accuracy and AUC-ROC comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Accuracy plot
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
bars1 = axes[0].bar(results_df['Model'], results_df['Accuracy'], color=colors, edgecolor='black')
axes[0].set_title('Model Accuracy Comparison', fontsize=13)
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0.80, 0.90)
axes[0].tick_params(axis='x', rotation=15)
for bar, val in zip(bars1, results_df['Accuracy']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')

# AUC-ROC plot
bars2 = axes[1].bar(results_df['Model'], results_df['AUC-ROC'], color=colors, edgecolor='black')
axes[1].set_title('Model AUC-ROC Comparison', fontsize=13)
axes[1].set_ylabel('AUC-ROC Score')
axes[1].set_ylim(0.70, 0.80)
axes[1].tick_params(axis='x', rotation=15)
for bar, val in zip(bars2, results_df['AUC-ROC']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Best model: Random Forest (highest accuracy)
best_model = models['Random Forest']
y_pred_best = best_model.predict(X_test)

# Detailed classification report
print("=== Classification Report (Random Forest) ===")
print(classification_report(y_test, y_pred_best, target_names=['No Maintenance', 'Maintenance Needed']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Maintenance', 'Maintenance Needed'],
            yticklabels=['No Maintenance', 'Maintenance Needed'])
plt.title('Confusion Matrix — Random Forest', fontsize=13)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 🎯 Step 9: Feature Importance

Which factors matter most when predicting if a train needs maintenance?

In [ ]:
# Feature importance from LightGBM (trained with more trees for better importance estimates)
lgbm_fi = LGBMClassifier(n_estimators=200, random_state=42, verbose=-1)
lgbm_fi.fit(X_train, y_train)

# Get feature importances and sort them
feat_imp = pd.Series(lgbm_fi.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=True).tail(15)

# Plot
plt.figure(figsize=(10, 7))
feat_imp.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Top 15 Most Important Features (LightGBM)', fontsize=13)
plt.xlabel('Feature Importance Score')
plt.tight_layout()
plt.show()

print("\nTop 10 features:")
print(feat_imp.tail(10).sort_values(ascending=False).to_string())

## ✅ Conclusion

### 🔑 What We Did
We built a **Predictive Maintenance classifier** for Indian Railways using real-world sensor and operational data.

### 📊 Key Findings

| Model | Accuracy | AUC-ROC |
|---|---|---|
| Random Forest | ~84.9% | ~0.756 |
| XGBoost | ~84.7% | ~0.757 |
| LightGBM | ~84.9% | ~0.754 |
| Gradient Boosting | ~84.9% | ~0.751 |

All four models achieved **~85% accuracy**, which is solid for this type of classification problem.

### 🔍 Most Important Factors for Maintenance Prediction
1. **Rail wear** — worn rails are a top indicator
2. **Brake pressure & pad wear** — braking system condition matters a lot
3. **Wheel wear** — heavily worn wheels signal upcoming failure
4. **Battery voltage & sensor health** — electrical system integrity
5. **Bearing & axle temperature** — heat buildup signals mechanical stress

### 💡 Insights for Real-World Use
- The dataset is **imbalanced** (~69% no maintenance vs ~31% maintenance needed) — models naturally lean toward predicting the majority class
- For a safety-critical system like railways, **recall for the maintenance class** matters most — missing a needed maintenance is more costly than a false alarm
- Future improvements: SMOTE for class balancing, hyperparameter tuning with GridSearchCV, and ensemble stacking

### 🚀 What a Railway Engineer Can Do With This
Deploy this model on live sensor streams to get **early warnings** before failures occur — saving lives, reducing delays, and cutting repair costs.

---
> **Note:** Leakage columns (`failure_type`, `failure_severity`, `risk_score`, `delay_minutes`) were intentionally removed because they are only known *after* a failure occurs and would give artificially high accuracy.

---
*Thank you for reading! If you found this helpful, please give an upvote ⬆️*